# NB_patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per the Silver-to-Gold STTM. Full rebuild via CREATE OR REPLACE TABLE ... AS SELECT; LEFT JOINs from the driving table so no patient is dropped.

In [0]:
# Catalog placeholder from the STTM (<catalog>)
dbutils.widgets.text("catalog", "databricks_caresync")
catalog = dbutils.widgets.get("catalog")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"
print(target_table)

In [0]:
# Full rebuild: driving table = silver.patients, LEFT JOIN to hospitals, insurance_providers and per-patient lab aggregates
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                                AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count,
    MAX(test_date)                                          AS last_test_date
  FROM {source_schema}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  CAST(p.first_name AS STRING)                                           AS first_name,
  CAST(p.last_name AS STRING)                                            AS last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                               AS gender,
  CAST(p.city AS STRING)                                                 AS city,
  CAST(p.state AS STRING)                                                AS state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  CAST(h.hospital_name AS STRING)                                        AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                       AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                         AS last_test_date
FROM {source_schema}.patients p
LEFT JOIN {source_schema}.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {source_schema}.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id
""")

display(spark.table(target_table).orderBy("total_tests", ascending=False).limit(10))

In [0]:
print("=" * 80)
print("[STEP 5] Computing processed record count")
print("=" * 80)

# Gold table rebuilt in the previous cell
updated_df = spark.table(target_table)
record_count = updated_df.count()

print(f"[STEP 5 COMPLETE] Processed record count: {record_count}")

dbutils.notebook.exit(str(record_count))